# Multilingual speech–text intent router

Third implemented speech project: 14 banking intents in English, Spanish and French. Frozen Whisper and multilingual MiniLM encoders feed a trained fusion head. Compare audio-only, text-only and joint inputs on exactly the same splits.

The committed pilot obtained **95.24% fusion accuracy** on 126 test utterances, versus 93.65% text-only and 56.35% audio-only. The fusion advantage is uncertain. Source speaker IDs are unavailable; these are utterance splits. The supplied dataset transcript is an input; this notebook does not evaluate an end-to-end ASR system.

This notebook reproduces the experiment. Its cells have no prefilled training outputs; the executed artifacts are committed under `evidence/2026-11-19/intent/`. CPU execution is supported and requires no paid API. Model and dataset downloads require internet access and disk space. Run each stage once, or choose fresh output directories before rerunning.

In [ ]:
import os, sys, subprocess
from pathlib import Path
repo = Path('/content/speech-intent-lab') if Path('/content').exists() else Path.cwd() / 'speech-intent-lab'
if repo.exists():
    raise RuntimeError(f'{repo} already exists; select a fresh checkout directory.')
subprocess.run(['git', 'clone', '--branch', 'codex/siri-real-speech-lab-20261119',
                'https://github.com/ByteBoss123/Speech-Command-.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.8.0',
                '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-speech.txt'], check=True)
print('Checkout:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Prepare real paired recordings

The source revision is pinned. The code removes exact duplicated audio/transcripts, then selects 8 training, 3 validation and 3 test examples per language/intent. Source `english_transcription` and class names are not model inputs. All 3 languages appear in training.

In [ ]:
subprocess.run([sys.executable, '-m', 'speechlab.intent_data'], check=True)

## Frozen encoders, then trained classification heads

Feature extraction is independent per example. Training-only means/scales are fitted afterward. Each model uses the same seed, learning rate and validation selection rule. Checkpoint selection completes before test scoring.

In [ ]:
subprocess.run([sys.executable, '-m', 'speechlab.intent_features'], check=True)
subprocess.run([sys.executable, '-m', 'speechlab.intent'], check=True)

In [ ]:
import json
from IPython.display import display, Markdown
report = json.loads(Path('runs/intent-pilot/metrics.json').read_text())
lines = ['| Input | Accuracy | Macro F1 | Correct |', '|---|---:|---:|---:|']
for mode, result in report['models'].items():
    test = result['test']
    lines.append(f"| {mode} | {100*test['accuracy']:.2f}% | {test['macro_f1']:.4f} | {test['correct']}/{test['n']} |")
display(Markdown('\n'.join(lines)))
print('Fusion minus text:', json.dumps(report['fusion_minus_text'], indent=2))
print('Limitations:', '\n'.join(report['limitations']))

## Reload the exported head and run actual audio/text inference

Use the first held-out recording without selecting it based on prediction correctness. Inference loads the raw audio, re-encodes both inputs, and reloads the saved head. For your own example, supply a 1–30 second file and the corresponding native-language transcript. Output probabilities are not calibrated and unknown intents are not rejected.

In [ ]:
rows = [json.loads(line) for line in Path('data/minds14-pilot/manifest.jsonl').read_text().splitlines()]
example = next(row for row in rows if row['split'] == 'test')
print('Reference intent:', example['intent'], '| Language:', example['language'])
subprocess.run([sys.executable, '-m', 'speechlab.intent_predict',
                str(Path('data/minds14-pilot') / example['path']), '--transcript', example['text'],
                '--model', 'runs/intent-pilot/fusion'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests_speech', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'speechlab.verify_intent', 'evidence/2026-11-19/intent'], check=True)

## What can be claimed

Implemented and trained a multilingual audio–text intent classifier with measured baselines, saved weights and reproducible evaluation. This does not establish generative speech-LLM training, speaker-independent multilingual generalization, production readiness or a reliable fusion improvement.

Sources: [MInDS-14](https://huggingface.co/datasets/PolyAI/minds14), [paper](https://aclanthology.org/2021.emnlp-main.591/), [Whisper](https://huggingface.co/openai/whisper-tiny), [MiniLM](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). See `docs/MULTILINGUAL_INTENT.md` for the full protocol. Implementation was completed with AI assistance.